# TDWI Lab 3 Part 3: PR Review Automations

In this lesson you will configure a **Cursor Automation** for review and fix, run a **Cloud Agent** to add a Streamlit **Revenue Explorer**, walk through draft → ready → automation (and optional fix PRs), merge stacked PRs (newest first), then add a **`scripts/check.sh`** gate motivated by real dependency issues (e.g. Streamlit vs pinned pandas).

**Lab design note:** You are learning the general **agent-as-reviewer** pattern and how **Automations** wire it on GitHub. Cursor also ships dedicated features for this—**Bugbot** and **Approval Agents**—and other AI tools have equivalents. In production, explore those built-ins first; we use a custom Automation here so the pattern is portable and not tied to one product feature.

## Learning Objectives

By the end of this mini-lesson you will be able to:
- Create a GitHub PR-triggered Cursor Automation with **review and fix** behavior
- Use the **Pull request opened** trigger (ready PRs only—not draft creation) to avoid automation loops
- Run a Cloud Agent to add a Streamlit Revenue Explorer and walk through the full draft → ready → review (→ optional fix) flow
- Merge stacked PRs **newest first** (fix into its base branch, then parent into `main`)
- Add a **`scripts/check.sh`** deterministic gate (pip dry-run + pytest) and wire it into agent workflow (prompt, `AGENTS.md`, hooks)
- Contrast a **custom Automation** with provider-native review (e.g. Cursor **Bugbot**, **Approval Agents**)

## Prerequisites

- Completed [README.md](README.md) setup (fork, clone, local `.venv`, test push)
- Completed [LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb](LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb) (env/secrets on your fork)
- Completed [LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb](LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb): pipeline fixes merged to `main`, tests green
- Cursor plan with **Automations** enabled; GitHub connected to **your fork**

**Important:** Configure the automation on the **same GitHub repo where you open PRs** (your fork). Each student sets up their own automation, unless the instructor demos on a shared fork.

## Step 1: Understand the workflow

Lab 3 uses a deliberate handoff between humans and agents:

1. **Cloud Agent** implements code and opens a **draft PR** (Step 4).
2. **You** optionally pull the branch for detailed local review (Step 5)—this lab does that **before** agent review; many teams defer detailed human review until **after** agent review.
3. **You** trigger automated review when your setup calls for it (Step 6—in this lab, **Ready for review** on GitHub).
4. **Cursor Automation** reviews the PR, posts comments, and—if it finds **blocking errors only**—may implement fixes and open a **new draft PR**.
5. **You** do deeper human review (checkout, tests, diff)—often **after** agent review, before merge—and optionally run another agent review round on fix PRs.
6. **You** merge when satisfied (Step 7), then add deterministic gates (`scripts/check.sh`, Step 8).

Cursor defines two related triggers ([Automations docs](https://cursor.com/docs/cloud-agent/automations)):

| Trigger | When it fires |
|---------|----------------|
| **Draft opened** | A draft PR is created |
| **Pull request opened** | A non-draft PR is created **or** a draft is **marked ready for review** |

**For this lab, use Pull request opened only** (do **not** use **Draft opened**). Implementation and fix PRs stay **drafts** until you mark them ready—so the automation does not re-fire on its own fix PRs.

**Loop prevention (important):**

| Setup | Risk |
|-------|------|
| Trigger on **draft opened** + agent fixes issues | High—the automation re-fires on every new draft |
| Trigger on ready + fix **all** suggestions | High—there are always more suggestions to fix |
| **This lab:** ready only + fix **blocking errors** + open **draft** fix PR | Low—you control when the next pass runs |

Prompts are not deterministic; this is a tested best practice, not a guarantee.

## Step 2: Create the PR review automation

**Why build your own?** This step teaches the portable workflow—trigger, instructions, output—not the only way to get AI review on PRs. For day-to-day work on Cursor, **Bugbot** is an excellent default; **Approval Agents** are another productized option. After the lab, try those alongside or instead of a custom Automation. See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) Recipe 5.

1. Open Cursor **Settings** → **Automations** (or the Automations section on [cursor.com](https://cursor.com)). See the [Automations documentation](https://cursor.com/docs/cloud-agent/automations) if the UI differs slightly.
2. Click **Create automation** (or equivalent).
3. **Trigger:** GitHub → **Pull request opened**.
4. **Repository:** Select **your fork** of this starter repo (e.g. `your-username/tdwi-agentic-sales-pipeline-starter`).
5. **Tools / output:** Enable **Comment on pull request** and any repo/PR tools the UI requires to open a **draft PR** when blocking fixes are needed.
6. Paste the following into the automation **instructions** field (general checklist—reuse on any PR in this repo):

```text
Review this pull request.

Focus on:
- Summary of what changed and whether the approach fits the existing codebase
- Correctness, edge cases, and error handling in the diff
- Whether tests were added or updated; note if the PR does not mention test results
- New or changed dependencies (e.g. requirements.txt): necessity and version pinning
- Scope: flag unrelated refactors or drive-by changes
- Security or data-handling concerns if relevant

Post a concise review as PR comments: summary, strengths, blocking errors, suggested improvements. Do not merge or approve. If you find blocking errors, implement the fixes and open a draft PR using this PR's branch (the branch under review) as the base—not the default branch. The fix PR must be a draft. Include a link to the fix PR in your review summary comment.
```

7. Save the automation.

## Step 3: Save and verify the automation

- Confirm the automation appears in your Automations list and is enabled for your fork.
- Use the dashboard **run history** (if available) after Step 6 to confirm it executed.

**Note:** If you already marked a Part 2 PR as ready, toggling ready again may not re-fire the trigger. Part 3’s new PR (Step 4) is the intended test.

## Step 4: Cloud Agent — add Revenue Explorer

Your fork should already have the [`AGENTS.md`](AGENTS.md) you updated in Part 2 on GitHub. The Cloud Agent reads that file automatically—you do **not** need to repeat everything in this prompt.

**Already in `AGENTS.md` (confirm it is pushed to your fork):**
- Repo context and layout (`generate_sales_report.py`, tests, data paths)
- **Testing workflow** — run `python -m pytest test_sales_report.py`, iterate until green before pushing
- **Lab boundaries** — do not modify README or lab notebooks

This prompt states only **what to build** for Part 3. If you changed `AGENTS.md` locally since Part 2, commit and push before starting the agent (same as Part 2 Step 4).

Start a Cloud Agent on **your fork** (same Dockerfile-managed environment as Part 1). Go to [cursor.com/agents](https://cursor.com/agents) or the Agents window in Cursor, select your repository, and paste this prompt:

```text
Add a small Streamlit app called revenue_explorer.py at the repo root.

Product requirements:
- Sidebar: date range filter, multi-select product, optional customer_id filter.
- Main area: KPIs (total revenue, order count, average order value) for the filtered data.
- One chart: daily revenue trend for the filtered data.

Do not duplicate logic. Write clean, well-organized code.
Open a PR when done.
```

### Watch the agent and open the PR on GitHub

1. In the [Agents dashboard](https://cursor.com/agents) (or the Agents panel in Cursor), open your session and follow progress until the run **finishes**—edits, terminal output, pytest runs. This may take several minutes.
2. When the agent completes, it will usually open a **draft PR** on your fork. That is expected; you do not need a non-draft PR at this step.
3. On GitHub, open **your fork** → **Pull requests** → the agent's PR (often a `cursor/...` branch into `main`). Confirm it shows **Draft**.
4. Continue to **Step 5** for local review before automation (a lab choice—see Step 5). You will mark the PR **Ready for review** in Step 6 to trigger your automation.

## Step 5: Local review before triggering automation (optional)

**Lab vs practice:** Many teams **defer detailed human review** until after agent review—not one fixed GitHub workflow, but a common pattern: let automated review (CI and agent) run first, then invest human time on the diff. **This lab runs local review before Step 6** to reinforce Part 2 and to surface a common agent mistake early (`pytest` green, `pip install` broken). After the workshop, you might do your detailed checkout and test pass after agent review instead.

1. From the draft PR you opened in Step 4, note the **branch name**.
2. Locally, fetch and check out the agent branch:

```bash
git fetch origin
git checkout <agent-branch-name>
```

3. With `.venv` activated, run tests:

```bash
python -m pytest test_sales_report.py
```

4. **Install dependencies** (this command may **fail**; read the following **warning** first).

   **WARNING**: An agent may pin **Streamlit** to a version that conflicts with pinned **pandas** in `requirements.txt`—**`pytest` can pass while `pip install` fails** with a resolver error. That is the bug we formalize in **Step 8**. If you see it, note it and continue to **Step 6**; you do not need to fix it yet.

```bash
pip install -r requirements.txt
```

5. Run the new app:

```bash
streamlit run revenue_explorer.py
```

6. Read the code changes. Do **not** merge yet—you will trigger the automation in **Step 6**.

**Skipping Step 5?** You may skim the diff on GitHub and continue to **Step 6**, then do detailed local review after agent review (before merge in **Step 7**). You will still hit the dependency lesson in **Step 8** if `pip install` was never run.

## Step 6: Mark the PR ready for review

Return to the **same PR** on GitHub.

1. Scroll to the **bottom** of the PR page and click **Ready for review** (shown on draft PRs only).
2. This fires **Pull request opened**—your review automation should start within a few minutes. You can also watch run status in the Cursor **Automations** run history (Step 3).

### When the automation finishes

Return to this PR on GitHub and review what the automation did:

1. Open the **Conversation** tab. Read the automation’s **review comments** (summary, strengths, blocking errors, suggestions).
2. Check whether it opened a **fix draft PR**. Your automation instructions tell it to do this only when it finds **blocking errors**—if the PR looked fine, you may see comments only and no new PR. That is normal.
3. If a fix draft PR exists, it is usually linked in the review comment or listed under **Pull requests** on your fork (new `cursor/...` or similar branch). It stays a **draft** and does **not** re-trigger your automation until you mark it ready.
4. **Detailed human review (common placement):** If you skipped Step 5, do your local pass now—checkout, `pytest`, `pip install -r requirements.txt`, read the diff—before merge. Agent review comments are input; you still own the merge decision.
5. **Optional: another review round.** After you review the fix locally (or skim the diff on GitHub), open the **fix PR**, scroll to the bottom, and click **Ready for review**. That triggers the same **review and fix** automation again—comments on the fix PR, and possibly another fix draft PR if it still finds blocking errors. You control when each round runs.

**Optional next steps:**

- Check out a fix branch locally, run tests and `pip install -r requirements.txt`, and review the diff before marking a fix PR ready.
- If **Bugbot** or **Approval Agents** are enabled, compare their output with your custom automation.
- Continue to **Step 7** to merge PRs when you are satisfied with the review rounds.

## Step 7: Merge stacked PRs (newest first)

When you are done with review rounds (Step 6), merge the PRs from this lab on GitHub.

1. Open **Pull requests** on your fork. List every open PR from Part 3—the Streamlit implementation PR (**into `main`**) and any **fix** PRs your automation created (**into the Streamlit branch**, not `main`—see Step 2).
2. **Merge newest first (stack order):** Each fix PR merges **into its base branch**. Merge fix PRs before the parent PR that targets `main`. Typical two-PR case: merge the **fix PR first** (fix → Streamlit branch), then the **Streamlit PR** (Streamlit branch → `main`). If you triggered another automation round, merge the newest PR first and work outward until the root PR merges to `main`. Sorting by **created** time (newest first) usually matches this order.
3. For each PR still in **Draft**, click **Ready for review** at the bottom if GitHub requires it before merge. Then **Merge pull request** when you are satisfied. Marking a draft ready can re-fire your automation—merge when you do not need another review round.
4. Locally, sync your default branch:

```bash
git checkout main
git pull origin main
```

**Dependency check:** If `pip install` failed in Step 5 (or you skipped Step 5), run `pip install -r requirements.txt` on `main` after the pull—you may see the same Streamlit/pandas conflict. **Step 8** adds a script so that check is never optional.

## Step 8: Add `scripts/check.sh` (Recipe 1 starter)

A single **check script** is a cheap deterministic gate—same idea as a human engineer running `make test` before push. This starter catches **dependency conflicts** with `pip install --dry-run` before you rely on a broken environment.

**Intentional lab ordering:** In production, this script runs **before** merge (see Step 9). We add it **after** Step 7 so you have already felt the gap: `pytest` alone is not enough, and even careful humans may skip `pip install` unless the gate is scripted. Whether you caught the Streamlit/pandas conflict in Step 5, after Step 6, or only on merged `main`, `check.sh` makes the dry-run non-optional.

1. Create the folder and file `scripts/check.sh` at the repo root (same level as `generate_sales_report.py`).
2. Paste the script below, then make it executable:

```bash
chmod +x scripts/check.sh
```

3. With `.venv` activated, run from the repo root:

```bash
bash scripts/check.sh
```

If Streamlit and pandas conflict, the **pip dry-run** step should fail with a resolver error—the same failure you may have seen at `pip install` in Step 5. That is the gate working.

```bash
#!/usr/bin/env bash
# TDWI lab — deterministic checks before push/PR (Recipe 1 starter).
# Run from repo root: bash scripts/check.sh

set -euo pipefail
cd "$(dirname "$0")/.."

echo "==> pip dry-run (catch dependency conflicts before install)"
python -m pip install --dry-run -r requirements.txt

# --- Extension points (uncomment as you grow this script) ---
# echo "==> ruff"
# ruff check .
# ruff format --check .

echo "==> pytest"
python -m pytest test_sales_report.py

echo "All checks passed."
```

4. Commit and push `scripts/check.sh` to your fork when it passes (or commit a failing state first if you will fix it in Step 10).

See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) Recipe 1 and the fuller templates in [`examples/scripts/`](examples/scripts/) for post-lab ideas.

## Step 9: Wire checks into every agent run

**Treat agents like engineers with tools**—you would not rely on a human to remember tests only sometimes. The same check script should run **before every push or PR**, not only in a one-off demo.

### Update the Streamlit prompt (for next time)

Step 4 ran **before** `check.sh` existed. For future Cloud Agent tasks, add one line to the **Engineering** section of your prompt:

```text
Before opening a PR, run bash scripts/check.sh and fix any failures.
```

### Optional: `AGENTS.md`

Under **Core Workflow Rules**, you can add:

```markdown
Before pushing or opening a PR, run `bash scripts/check.sh` and fix all failures. Re-run until exit code 0.
```

Commit and push if you add this (same as Part 2).

### Other surfaces (same script, different triggers)

| Surface | Role |
|---------|------|
| **Agent prompt** | Explicit instruction each session |
| **`AGENTS.md` / rules** | Repo policy every Cloud Agent reads |
| **Pre-commit / git hooks** | Block commit or push locally |
| **IDE harness hooks** | Cursor and GitHub Copilot support hooks that run commands around agent actions; Claude Code has hooks as well |
| **GitHub Actions** (Recipe 4) | Same script on every PR—see [`examples/github/workflows/ci.yml`](examples/github/workflows/ci.yml) |

Humans use the same pattern: local hooks or habit before push, plus CI on the PR. Agents get the script via prompt, `AGENTS.md`, or hooks—not by replacing the script with judgment calls.

## Step 10: Cloud Agent — run `check.sh` and fix (demo only)

**Lab demo:** If `check.sh` failed after Step 8 (often a Streamlit/pandas pin conflict), start **one more** Cloud Agent to run the script and fix `requirements.txt` or other issues.

**Production pattern:** You normally **do not** spin up a separate Cloud Agent only to run CI. The check script runs inside **every** implementation session—via prompt, `AGENTS.md`, pre-commit, or harness hooks—so the agent fixes issues before opening a PR.

Paste this prompt:

```text
Run bash scripts/check.sh from the repo root. Fix any failures (especially requirements.txt if pip dry-run reports dependency conflicts). Re-run the script until it passes. Open a draft PR when done. Do not modify README.md or any lab notebook.
```

Watch the agent in the [Agents dashboard](https://cursor.com/agents) as in Step 4. Review and merge the PR when green—**Step 7** merge order applies if you still have multiple open PRs.

## Further reading: workflow framework

Parts 1–3 practiced pieces of a larger pattern: **probabilistic agents** for implementation, **deterministic scripts and CI** for gates, **fresh-context final review** (another agent—not the implementer) before humans merge. Review while building (tests, iteration) is expected; the framework adds a deliberate **final** critique—like asking a colleague with fresh eyes.

See [WORKFLOW_RECIPES.md](WORKFLOW_RECIPES.md) for the full framework, **[Harness vs team workflow](WORKFLOW_RECIPES.md#harness-vs-team-workflow)**, **Recipe 1** (`scripts/check.sh`—introduced in Step 8), Recipes 2–6, and example files in [`examples/`](examples/).


## Debrief questions

1. Why use a **draft PR** for implementation agents and **ready for review** for the automation?
2. Why does this lab fix **blocking errors only** (not every suggestion)? What can go wrong if you trigger on **draft opened** and ask the automation to fix all suggestions?
3. What did your automation catch that you would have missed? What did it miss?
4. When would you use a **custom Automation** vs **Bugbot** vs **Approval Agents** vs both?
5. When do you do **detailed** human review on an agent PR—before agent review (Step 5) or after (before merge)? What are the tradeoffs?
6. Why merge stacked PRs **newest first** (fix into its base branch, then parent into `main`)?
7. What did `pip install --dry-run` catch that `pytest` alone did not?
8. Why run `check.sh` on **every** agent session (prompt / `AGENTS.md` / hooks) instead of a second Cloud Agent only when something breaks?
9. How could you add a **CI completed** trigger so review runs only after green checks?
10. Why does the lab teach custom Automations if vendor built-ins exist?